# Word Embedding Models: word2vec #

### Everything gets a vector! ###

We've already been exploring vectors involving words: consider scikit-learn's `CountVectorizer()`, for example, which we used to create the document-term matrix for our tf-idf calculations. That looked at words in relation to the documents in which they appeared.

Today, however, we're going to look at words in relation to all other words in a corpus. The vectors that describe these types of relations are called, appropriately enough, *word vectors*. (And sometimes also *word embeddings*).

### What is a word vector? ###

A *word vector* or *word embedding* is a numerical representation of a word within a corpus, based on co-occurence with other words. Linguists have found that much of the meaning of a word can be derived from looking at the context in which it appears. (In linguistics, this is known as the theory of *distributional semantics*).

### What is Word2Vec? ###

Word2vec is one popular approach to representing words in this numerical format. Conveniently, word2vec is implemented in a library called `gensim`.

Word2Vec is a *neural-network* or *deep learning* based approach of generating word vectors.

There are many resources out there that will go into the heavy details of deep learning in general or deep learning for NLP such as Yoav Goldberg's Neural Network Methods in Natural Language Processing (Morgan & Claypool Publishers, 2017). Today, you'll get a high level overview -- just enough for you to understand what w2v is doing.

# Let's try it out!

## Install gensim


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 27.8 MB/s eta 0:00:00


## Import gensim, nltk tokenizers, glob, and Path

In [ ]:
import gensim #

# and some other stuff
from nltk.tokenize import sent_tokenize
from nltk.tokenize.treebank import TreebankWordTokenizer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import glob
from pathlib import Path

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## Load in our corpus

In [ ]:
# For downloading large files from Google Drive
# https://github.com/wkentaro/gdown
import gdown

# then download the zip files
# atlanta
gdown.download('https://drive.google.com/uc?export=download&id=1gIm9NcoeY1gn9EQjRr2MojGRJ-fpBSqz', quiet=False)

# unzip it
!unzip Atlanta-random.jsonl.zip

Downloading...
From: https://drive.google.com/uc?export=download&id=1gIm9NcoeY1gn9EQjRr2MojGRJ-fpBSqz
To: /content/Atlanta-random.jsonl.zip
100%|██████████| 17.1M/17.1M [00:00<00:00, 41.4MB/s]


Archive:  Atlanta-random.jsonl.zip
  inflating: Atlanta-random.jsonl    


### Process the docs

As a first step, we'll need to create a list of all the reviews in the `Atlanta-random.jsonl` file, with each review stored as a single string. This is the same exact code we used last class, but condensed a little.

In [ ]:
# import libraries
import os             # for directory/file manipulation
import json           # for json
import pandas as pd   # for dataframes
import textwrap       # for nice formatting

# read in the file
atlanta_reviews_df = pd.read_json(path_or_buf="./Atlanta-random.jsonl", lines=True)

# first extract the 'comment' values from the dataframe
comments = atlanta_reviews_df['comment'].tolist()

# create list to store reviews
all_reviews = []

# iterate through the comments and append the reviews to the list
for comment in comments:
  all_reviews.append(comment['text'])

# print out the length just to check that everything got in
len(all_reviews)

34370

We learned last class that there were some extra HTML tags embedded in the review text, and we also saw some hex codes. Let's see if we can clean things up a little before we move further.

In [ ]:
from bs4 import BeautifulSoup

# new array w/ clean text
all_reviews_clean = []

for review in all_reviews:
    soup = BeautifulSoup(review, "html.parser")
    text = soup.get_text(separator=' ')

    all_reviews_clean.append(text)

# print out first one just to check
print(textwrap.fill(all_reviews_clean[0], 100))

Very overdue to try this local spot but was pleasantly surprised. It is nice inside comparable to
any downtown atl hookah and drinks bar. Tons of beer specials on game days. Menu is limited but
something for everyone. I had the whiting, fries and hush puppies and they were tasty. Attentive
service as well. Hookah was $40 and $10 for refills so it was a HARD PASS for me. That is nightclub
prices and definitely unexpected in a local bar. They should rethink that price structure,
especially on slower weeknights.


One last thing before we get started. Let's create some legible IDs for each revivew using some other info that's in the dataframe.

In [ ]:
# extract the 'business' values from the dataframe
businesses = atlanta_reviews_df['business'].tolist()

# create list to store business aliases
aliases = []

# iterate through the business and append the alias to the list
for business in businesses:
  aliases.append(business['alias'])

# extract ratings
ratings = atlanta_reviews_df['rating'].tolist()

# create list to store IDs <-- we'll use list this going forward
ids = []

# now put them all together into IDs
for i, alias in enumerate(aliases):
  id = alias + "-review" + str(i) + "-" + str(ratings[i]) + "stars"
  ids.append(id)

# print out the first one to check
ids[0]

'harolds-bar-and-grill-atlanta-3-review0-4stars'

Next, we need to get each of the docs in our `all_reviews_clean` list into the format required by gensim's implementation of word2vec.

We know from the gensim documentation (and also common sense) that the input to `word2vec` is sentences. So let's define a function that a takes a list of texts (e.g. our `all_reviews_clean::` list) and converts it into sentences for gensim word2vec to use. The function will lower-case text and tokenize by sentence and word. It will also print out a count of the sentences in each doc, so that we get some sort of status indicator that it's parsing the sentences correctly.

In [ ]:
# need our handy nltk tokenizer
tokenizer = TreebankWordTokenizer()

# and the function
def make_sentences(list_txt):
    all_txt = []
    counter = 0
    for txt in list_txt:
        lower_txt = txt.lower()
        sentences = sent_tokenize(lower_txt)
        sentences = [tokenizer.tokenize(sent) for sent in sentences]
        all_txt += sentences
        print("ID: " + ids[counter]) # let's print the id of the review
        print("Sentences: " + str(len(sentences)))  # let's check how many sentences there are per article
        counter += 1
    return all_txt

In [ ]:
# now let's run it
sentences = make_sentences(all_reviews_clean)

Streaming output truncated to the last 5000 lines.
ID: ruths-chris-steak-house-buckhead-atlanta-buckhead-3-review31870-1stars
Sentences: 3
ID: ruths-chris-steak-house-buckhead-atlanta-buckhead-3-review31871-5stars
Sentences: 32
ID: ruths-chris-steak-house-buckhead-atlanta-buckhead-3-review31872-5stars
Sentences: 38
ID: ruths-chris-steak-house-buckhead-atlanta-buckhead-3-review31873-3stars
Sentences: 14
ID: ruths-chris-steak-house-buckhead-atlanta-buckhead-3-review31874-5stars
Sentences: 4
ID: wildleaf-salads-atlanta-review31875-4stars
Sentences: 8
ID: wildleaf-salads-atlanta-review31876-1stars
Sentences: 7
ID: wildleaf-salads-atlanta-review31877-2stars
Sentences: 3
ID: wildleaf-salads-atlanta-review31878-5stars
Sentences: 8
ID: wildleaf-salads-atlanta-review31879-5stars
Sentences: 9
ID: wildleaf-salads-atlanta-review31880-1stars
Sentences: 9
ID: wildleaf-salads-atlanta-review31881-3stars
Sentences: 6
ID: wildleaf-salads-atlanta-review31882-1stars
Sentences: 9
ID: wildleaf-salads-atlant

## Train model

Now that we have our corpus ready for gensim, we can train the model. To do so, we call the function `gensim.models.Word2Vec()`. This function has a couple dozen parameters, some of which are more important than others.

Here are a few major ones. Only two are MANDATORY: these are marked with an asterisk:

1. `sentences*`: This is where you provide your data. It must be in a format of iterable of iterables.
2. `sg`: Your choice of training algorithm. There are two standard ways of training W2V vectors -- 'skipgram' and 'CBOW'. If you enter 1 here the skip-gram is applied; otherwise, the default is CBOW.
3. `size*`: This is the length of your resulting word vectors. If you have a large corpus (>few billion tokens) you can go up to 100-300 dimensions. Generally word vectors with more dimensions give better results.
4. `window`: This is the window of context words you are training on. In other words, how many words come before and after your given word. A good number is 4 here but this can vary depending on what you are interested in. For instance, if you are more interested in embeddings that embody semantic meaning, smaller window sizes work better.
5. `alpha`: The learning rate of your model. If you are interested in machine learning experimentation with your vectors you may experiment with this parameter.
6. `seed` (int): This is the random seed for your random initialization. All deep learning models initialize the weights with random floats before training. This is a useful field if you want to replicate your experiments because giving this a seed will initialize 'randomly' deterministically.
7. `min_count`: This is the minimum frequency threshold. If a given word appears with lower frequency than provided it will be ignored. This is here because words with very low frequency are hard to train.
8. `epochs`: This is the number of iterations (entire run) over the corpus, also known as epochs. Default is 5. Usually anything between 1-10 is ok. The trade offs are that if you have higher iterations, it will take longer to train and the model may overfit on your dataset. However, longer training will allow your vectors to perform better on tasks relevant to your dataset.

Most of these settings will not concern us. As you'll see below, we are only going to use four arguments.

\* On newer versions of gensim, `size` has changed to `vector_size`. But Colab has not yet updated theirs so `size` still works.

In [ ]:
# let's train our model!
atl_reviews_model = gensim.models.Word2Vec(
    sentences,
    min_count=2, # default is 5; this trims the corpus for words only used once;
    vector_size=100,
    workers=5) # parallel processing; needs Cython

Hooray! We have a trained word2vec model: `atl_reviews_model`!

### Save model — and load it

It's often useful to save your trained model to disk so that you can reload it as needed.

In [ ]:
# how to store the above files to Google Drive

from google.colab import drive
drive.mount('/content/gdrive')

atl_reviews_model.save('/content/gdrive/My Drive/atl_reviews_model')

Mounted at /content/gdrive


And you can load an old model in the same way

In [ ]:
# how you would load an old model from your own google drive
old_model = gensim.models.Word2Vec.load('/content/gdrive/My Drive/atl_reviews_model')

## Let's play!

All the word vectors can be accessed for any trained model as follows:

In [ ]:
atl_reviews_model.wv.vectors # These are the vectors from inside the model

Check to see how many there are. What do you think this count should be? (Remember that we removed all words that appear only once.)

In [ ]:
# number of vectors
len(atl_reviews_model.wv.vectors)

In [ ]:
# check the length of the first vector
print("Length of the first vector: " + str(len(atl_reviews_model.wv[0])))

# check the length of a random one in the middle
print("Length of random vector: " + str(len(atl_reviews_model.wv[2097])))

The corresponding words to each vector can be accessed using this dictonary:


In [ ]:
print(atl_reviews_model.wv.key_to_index)

The words in order of frequency can be accessed using this dictionary:

In [ ]:
import textwrap

# print all words
# words_list = atl_reviews_model.wv.index_to_key

# print only the top 300
words_list = atl_reviews_model.wv.index_to_key[:300]

formatted_words = textwrap.fill(', '.join(words_list), width=80)
print(formatted_words)

In [ ]:
word_of_interest = "coffee"

print (f"Row number (index): {atl_reviews_model.wv.key_to_index[word_of_interest]}")

print(f"Length of vector: {len(atl_reviews_model.wv.vectors[atl_reviews_model.wv.key_to_index[word_of_interest]])}")

print (f"Word vector: {atl_reviews_model.wv.vectors[atl_reviews_model.wv.key_to_index[word_of_interest]]}")

### Similarity

word2vec can tell us which words, according to its model, are most similar to any other. We call `model.wv.most_similar("word", topn=number of similar words)`. Let's try "delicious."

In [ ]:
# basic similarity w/ adjectives
atl_reviews_model.wv.most_similar("coffee", topn=10)

In [ ]:
# basic similarity w/ nouns
atl_reviews_model.wv.most_similar("fries", topn=10)

### Similarity between two words

We can also choose specific words to compare, with similarity reported as cosine similarity.

As discussed, identical vectors will have a score of 1. Totally unrelated vectors will have scores close to 0, meaning that they are *orthogonal*.

In [ ]:
# similarity b/t two words

print(atl_reviews_model.wv.similarity(w1="soup",w2="delicious"))
print(atl_reviews_model.wv.similarity(w1="soup",w2="disgusting"))

## Pretrained models

One of the nice things about vector models (and many language models, including LLMs, is that they can be pretrained and saved, and then made more widely available, so that you don't need to re-train your model each time from scratch. There are also other reasons why pretrained models are helpful or desirable. What are some other reasons that you can think of?

`gensim` makes a bunch of pretrained models available for download. You can see what's in the current list like so:

In [ ]:
import gensim.downloader
for model_name in gensim.downloader.info()['models'].keys():
    print(model_name)

Let's pick `glove-wiki-gigaword-100`. The [GloVe](https://nlp.stanford.edu/projects/glove/) project stands for Global Vectors for Word Representations, rn by a team at Stanford. This particular model was trained on a dump of Wikipedia from 2014, and Gigaword 5th Edition, which is a large dataset of English newswire text compiled from sources like the Associated Press and The New York Times. There are about 6 billion tokens, but each vector has only 100 dimensions, just like our Atlanta reviews model.

While it is downloading, I am going to give you a quick lecture on [this research paper](https://aclanthology.org/N13-1090/) and [this one](https://gargnikhil.com/files/pdfs/GSJZ18_embedstereotypes.pdf), and a few more interesting things about vectors that they popularized.

In [ ]:
## note to self -- use smaller model or cache per claude rec

glove_vectors = gensim.downloader.load('glove-wiki-gigaword-100')

## Similarlity with a pretrained model

In [ ]:
# almost the same as our model, except withou the '.wv' in the middle
# because what we've downloaded is not the whole model, but rather just
# the vectors
glove_vectors.most_similar("delicious", topn=10)

## Similarity between two words

As above, very similar to before, except no `.wv` in the middle:

In [ ]:
# similarity b/t two words

print(glove_vectors.similarity(w1="soup",w2="delicious"))
print(glove_vectors.similarity(w1="soup",w2="disgusting"))

Comprehension check: Why are these numbers different than the ones we got before?

### Analogy

Here we go! As explained in the mini-lecture, because vectors can be added and subtracted from each other, we can set up analogies. The most commonly seen example is:

'Man is to King as Woman is to ____?'

We can implement this method with a built-in function. The signs come straight out of the arithmetic. `positive` and `negative` are just labels for which vectors get added and which get subtracted:

`king - man + woman`

The order of the terms as they are written can be a little confusing, so another way to understand this is as a journey through the vector space. Start at **king**. Subtract **man** — you strip out whatever makes king male, leaving something like "royalty, gender removed." Add **woman** — you put a gender back in, but the other one. Where you land is near **queen**.

So the sign of each word is determined by which side of the analogy it sits on, not by anything about the word itself:

Here it is a chart of the same thing:

| Word | Role in the analogy | Sign |
|---|---|---|
| `man` | the term you're moving **away** from | negative |
| `woman` | the term you're moving **toward** | positive |
| `king` | the starting point you're transforming | positive |

In [ ]:
# analogies
# format is: "man is to king as woman is to ???"

results = glove_vectors.most_similar(positive=['woman', 'king'],
                                     negative=['man'],
                                     topn=10)

for word, score in results:
    print(f"{word}: {score:.4f}")

## Exercise:

**Use the code below, swapping out the relevant words, to see if you can replicate some of the findings from the "Stereotypes" paper.**

In [ ]:
# analogies
# format is: "man is to king as woman is to ???"

results = glove_vectors.most_similar(positive=['woman', 'king'],
                                     negative=['man'],
                                     topn=10)

for word, score in results:
    print(f"{word}: {score:.4f}")

## BONUS: Visualization!

Find below some code you can use to make visualizations from your word2vec model. We can't visualize all the many dimensions in our model, so we need to reduce them to two dimensions for our meager human brains. We do that with something called principal component analysis (PCA).

Don't worry about the details for now. This is just a fun way to take a look at the output of our model.

**Remember**: Our visualization reduces MANY dimensions to two, so a lot of information is lost.

In [ ]:
### Let's do some visualization ###

import numpy as np

# Get the interactive Tools for Matplotlib
# %matplotlib notebook # doesn't work for colab
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.decomposition import PCA

from gensim.test.utils import datapath, get_tmpfile
from gensim.models import KeyedVectors

In [ ]:
def display_pca_scatterplot(model, words=None, sample=0):
    if words == None:
        if sample > 0:
            words = np.random.choice(list(atl_reviews_model.wv.vocab.keys()), sample)
        else:
            words = [ word for word in atl_reviews_model.wv.vocab ]

#    word_vectors = np.array([model[w] for w in words]) <-- gensim 3 version
    word_vectors = np.array([model.wv[w] for w in words]) # gensim 4 version

    twodim = PCA().fit_transform(word_vectors)[:,:2]

    plt.figure(figsize=(6,6))
    plt.scatter(twodim[:,0], twodim[:,1], edgecolors='k', c='r')
    for word, (x,y) in zip(words, twodim):
        plt.text(x+0.05, y+0.05, word)

In [ ]:
display_pca_scatterplot(atl_reviews_model, ['italian','french','american','korean','japanese','mexican','chinese'])

# display_pca_scatterplot(ccp_model, sample=20)

## Exercise 3a:

**Copy the code above and plot some words that you think might be similar or different from each other.**

In [ ]:
# your plot here

## Exercise 3b:

**What do you think the plot shows you about the words? Did they confirm or contradict what you though they would show?**

In [ ]:
# your answer here

In [ ]:
# @title BONUS 2: Bias calculation (check slide deck for explanation)

word_of_interest = "exotic" # @param {"type":"string","placeholder":"australia"}
bias_word1 = "italian" # @param {"type":"string","placeholder":"she"}
bias_word2 = "indian" # @param {"type":"string","placeholder":"he"}


bias_direction = atl_reviews_model.wv[bias_word1] - atl_reviews_model.wv[bias_word2]
bias_magnitude = atl_reviews_model.wv.cosine_similarities(atl_reviews_model.wv[word_of_interest], [bias_direction])
print (bias_magnitude)


*Lauren F. Klein wrote version 1.0 of this notebook in 2019 based on the [Advanced Topics in Word Vectors workshop](https://dh2018.adho.org/en/machine-reading-part-ii-advanced-topics-in-word-vectors/) at DH 2018 as well as tutorials by [Radim Rehurek](https://rare-technologies.com/word2vec-tutorial/) and [Chris McCormick](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/) and class notebooks by Sandeep Soni and Maria Antoniak. It was updated again in 2021, 2022, 2024, and 2026.*